In [39]:
# FULL INTEGRATION OF BACKWARD-MODEL PIPELINE (V7 - KAGGLE VERIFIED)
# This script combines the notebook base with Laplace Uncertainty Estimation.
# Optimized for 500 stars and fixed shape broadcasting errors.

import os
import time
import json
import numpy as np
import h5py
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

tfb = tfp.bijectors

# ================================================================
# 1. CONFIGURATION
# ================================================================

class Config:
    # KAGGLE PATHS (DO NOT CHANGE)
    H5_PATH    = "/kaggle/input/datasets/aneeshshastri/aspcapstar-dr17-150kstars/apogee_dr17_parallel.h5"
    STATS_PATH = "/kaggle/input/models/aneeshshastri/stargen-comparision/tensorflow2/default/15/dataset_stats_120k.npz"
    MODEL_PATH = "/kaggle/input/models/aneeshshastri/stargen-comparision/tensorflow2/default/15/ThePayne.keras"
    CNN_MODEL_PATH = "/kaggle/input/models/aneeshshastri/backward-warmstart/tensorflow2/default/1/cnn_label_predictor_inner (1).keras"
    CNN_STATS_PATH = "/kaggle/input/models/aneeshshastri/backward-warmstart/tensorflow2/default/1/cnn_label_stats.npz"
    APOGEE_MASK_PATH = "/kaggle/input/datasets/aneeshshastri/element-masks/apogee_inference_mask.npy"
    
    SELECTED_LABELS = [
        'TEFF', 'LOGG', 'M_H', 'VMICRO', 'VMACRO', 'VSINI',
        'C_FE', 'N_FE', 'O_FE', 'FE_H',
        'MG_FE', 'SI_FE', 'CA_FE', 'TI_FE', 'S_FE',
        'AL_FE', 'MN_FE', 'NI_FE', 'CR_FE', 'K_FE',
        'NA_FE', 'V_FE', 'CO_FE',
    ]
    ABUNDANCE_INDICES = [i for i, l in enumerate(SELECTED_LABELS) if '_FE' in l]
    FE_H_INDEX = SELECTED_LABELS.index('FE_H')
    N_LABELS_RAW = len(SELECTED_LABELS)
    BADPIX_CUTOFF = 1e-3
    RANDOM_SEED   = 42
    BATCH_SIZE_STARS = 32
    PRIOR_WEIGHT  = 1.0
    TOTAL_STARS   = 500
    
    TEST_START = 140_000
    TEST_END   = 149_500
    
    # MAP hyper-params
    N_STEPS_CORE = 1200
    N_STEPS_ELEM = 400
    LR_CORE_INIT = 0.03
    LR_CORE_END  = 1e-4
    LR_ELEM_INIT = 0.03
    LR_ELEM_END  = 1e-4

    # Label groups
    CORE_LABELS = ['TEFF', 'LOGG', 'M_H', 'VMICRO', 'VMACRO', 'VSINI', 'C_FE', 'N_FE', 'O_FE']
    ABUND_LABELS = ['FE_H',
        'MG_FE', 'SI_FE', 'CA_FE', 'TI_FE', 'S_FE',
        'AL_FE', 'MN_FE', 'NI_FE', 'CR_FE', 'K_FE',
        'NA_FE', 'V_FE', 'CO_FE',]
    FE_H_INDEX   = SELECTED_LABELS.index('FE_H')
    ERRORS       = [f'{l}_ERR' for l in SELECTED_LABELS]

config = Config()
np.random.seed(config.RANDOM_SEED)
tf.random.set_seed(config.RANDOM_SEED)

CORE_INDICES  = [config.SELECTED_LABELS.index(l) for l in config.CORE_LABELS]
ABUND_INDICES = [config.SELECTED_LABELS.index(l) for l in config.ABUND_LABELS]
ABUNDANCE_INDICES = [config.SELECTED_LABELS.index(l) for l in config.SELECTED_LABELS if '_FE' in l]
N_PIXELS = 8575

In [40]:
# ================================================================
# 2. CUSTOM LAYERS
# ================================================================

@register_keras_serializable()
class ColumnSelector(layers.Layer):
    def __init__(self, indices, **kwargs):
        super().__init__(**kwargs)
        self.indices = list(indices)
    def call(self, inputs): return tf.gather(inputs, self.indices, axis=1)
    def get_config(self): c = super().get_config(); c.update({'indices': self.indices}); return c

@register_keras_serializable()
class BeerLambertLaw(layers.Layer):
    def __init__(self, **kwargs): super().__init__(**kwargs)
    def call(self, k, tau): return k * tf.math.exp(-tau)

@register_keras_serializable()
class GetAbundances(layers.Layer):
    def __init__(self, col_id, **kwargs):
        super().__init__(**kwargs)
        self.index = col_id
    def call(self, inputs): return inputs[:, self.index:self.index+1]
    def get_config(self): c = super().get_config(); c.update({'col_id': self.index}); return c

@register_keras_serializable()
class SparseProjector(tf.keras.layers.Layer):
    def __init__(self, active_indices, weights, total_pixels, label_name="unknown", **kwargs):
        kwargs['name'] = f"Sparse_Projector_{label_name}"
        super().__init__(**kwargs)
        self.total_pixels = total_pixels
        self.label_name = label_name
        self.active_indices = tf.constant(active_indices, dtype=tf.int32)
        self.weights_tensor = tf.constant(weights, dtype=tf.float32)
    def call(self, local_tau):
        local_tau_32 = tf.cast(local_tau, tf.float32)
        weighted = local_tau_32 * self.weights_tensor[None, :]
        batch_size = tf.shape(local_tau)[0]
        n_active = tf.shape(self.active_indices)[0]
        batch_ids = tf.repeat(tf.range(batch_size), n_active)
        pixel_ids = tf.tile(self.active_indices, tf.expand_dims(batch_size, axis=0))
        scatter_idx = tf.stack([batch_ids, pixel_ids], axis=-1)
        return tf.scatter_nd(scatter_idx, tf.reshape(weighted, [-1]),
                             shape=tf.stack([batch_size, tf.constant(self.total_pixels, dtype=tf.int32)]))
    def get_config(self):
        c = super().get_config()
        c.update({'active_indices': self.active_indices.numpy().tolist(),
                  'weights': self.weights_tensor.numpy().tolist(),
                  'total_pixels': self.total_pixels, 'label_name': self.label_name})
        return c
    @classmethod
    def from_config(cls, cfg):
        cfg['active_indices'] = np.array(cfg['active_indices'], dtype=np.int32)
        cfg['weights'] = np.array(cfg['weights'], dtype=np.float32)
        cfg.pop('name', None)
        return cls(**cfg)

@register_keras_serializable()
class HeteroscedasticCNNPredictor(tf.keras.Model):
    def __init__(self, n_labels=23, **kwargs):
        super().__init__(**kwargs)
        self.n_labels = n_labels
        self.reshape_in = layers.Reshape((N_PIXELS, 1))
        self.conv1 = layers.Conv1D(32, 16, strides=4, activation='relu', padding='same')
        self.bn1 = layers.BatchNormalization()
        self.conv2 = layers.Conv1D(64, 8, strides=4, activation='relu', padding='same')
        self.bn2 = layers.BatchNormalization()
        self.conv3 = layers.Conv1D(128, 4, strides=2, activation='relu', padding='same')
        self.bn3 = layers.BatchNormalization()
        self.conv4 = layers.Conv1D(256, 4, strides=2, activation='relu', padding='same')
        self.bn4 = layers.BatchNormalization()
        self.gap = layers.GlobalAveragePooling1D()
        self.dense1 = layers.Dense(512, activation='relu')
        self.drop1 = layers.Dropout(0.3)
        self.dense2 = layers.Dense(256, activation='relu')
        self.drop2 = layers.Dropout(0.2)
        self.mean_head = layers.Dense(n_labels, name='label_mean')
        self.log_var_head = layers.Dense(n_labels, name='label_log_var')
    def call(self, x, training=False):
        h = self.reshape_in(x)
        h = self.bn1(self.conv1(h), training=training)
        h = self.bn2(self.conv2(h), training=training)
        h = self.bn3(self.conv3(h), training=training)
        h = self.bn4(self.conv4(h), training=training)
        h = self.gap(h)
        h = self.drop1(self.dense1(h), training=training)
        h = self.drop2(self.dense2(h), training=training)
        return self.mean_head(h), self.log_var_head(h)
    def get_config(self):
        c = super().get_config(); c['n_labels'] = self.n_labels; return c

@register_keras_serializable()
class TrainableCNNPredictor(HeteroscedasticCNNPredictor): pass

@register_keras_serializable()
def chi2_estimate(y_true, y_pred):
    """Quick per-batch χ² estimate used as a Keras metric."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    if len(y_pred.shape) == 2:
        y_pred = tf.expand_dims(y_pred, -1)
    real_flux  = y_true[:, :, 0:1]
    ivar       = y_true[:, :, 1:2]
    valid_mask = tf.cast(real_flux > config.BADPIX_CUTOFF, tf.float32)
    safe_flux  = tf.where(valid_mask == 1.0, real_flux, y_pred)
    weight     = tf.where(ivar > 0, ivar, 0.0)
    wmse_term  = tf.square(safe_flux - y_pred) * weight * valid_mask
    n_valid    = tf.reduce_sum(valid_mask, 1)
    n_params   = len(config.SELECTED_LABELS)
    dof        = n_valid - n_params
    return tf.reduce_sum(wmse_term, 1) / dof


@register_keras_serializable()
def spl_loss(y_true, y_pred):
    """Spectral pixel loss — weighted MSE used during training."""
    if len(y_pred.shape) == 2:
        y_pred = tf.expand_dims(y_pred, -1)
    real_flux  = y_true[:, :, 0:1]
    ivar       = y_true[:, :, 1:2]
    valid_mask = tf.cast(real_flux > config.BADPIX_CUTOFF, tf.float32)
    safe_flux  = tf.where(valid_mask == 1.0, real_flux, y_pred)
    ivar_safe  = tf.clip_by_value(ivar / 1000.0, 0.0, 1.0)
    weight     = tf.where(
        (safe_flux < 0.9) & (ivar > 0),
        tf.maximum(ivar_safe, tf.cast(1.0, dtype=tf.float32)),
        ivar_safe,
    )
    wmse_term = tf.square(safe_flux - y_pred) * weight * valid_mask
    loss      = tf.where(tf.math.is_finite(wmse_term), wmse_term, tf.zeros_like(wmse_term))
    return loss


In [41]:
# ================================================================
# 3. GLOBAL BIJECTORS AND BOUNDS
# ================================================================

# Bounds from notebook
lower_bounds = [3000.,-0.5,-2.5,0.,0.,-20.,-2.5,-2.5,-2.5,-2.5,-2.0,-1.5,-1.5,-2.0,-2.0,-2.0,-2.0,-2.0,-2.5,-2.0,-2.0,-2.0,-2.0]
upper_bounds = [25000.,6.0,2.0,6.0,25.0,150.0,3.0,4.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,4.0,5.0,3.0,4.0]
bounds_low  = tf.constant(lower_bounds, tf.float32)
bounds_high = tf.constant(upper_bounds, tf.float32)

_core_low  = tf.gather(bounds_low, CORE_INDICES)
_core_high = tf.gather(bounds_high, CORE_INDICES)
_core_bij  = tfb.Chain([tfb.Shift(_core_low), tfb.Scale(_core_high - _core_low), tfb.Sigmoid()])
_core_inv  = tfb.Invert(_core_bij)

_joint_bij = tfb.Chain([tfb.Shift(bounds_low), tfb.Scale(bounds_high - bounds_low), tfb.Sigmoid()])
_joint_inv = tfb.Invert(_joint_bij)

In [42]:
# ================================================================
# 4. DATA PROCESSING UTILS
# ================================================================

def get_27_features(labels_23):
    idx = lambda l: config.SELECTED_LABELS.index(l)
    teff   = labels_23[:, idx('TEFF')]
    logg   = labels_23[:, idx('LOGG')]
    vmacro = labels_23[:, idx('VMACRO')]
    vsini  = labels_23[:, idx('VSINI')]
    c_fe   = labels_23[:, idx('C_FE')]
    o_fe   = labels_23[:, idx('O_FE')]
    m_h    = labels_23[:, idx('M_H')]
    eng = tf.stack([
        5040.0 / (teff + 1e-6),
        tf.sqrt(tf.square(vmacro) + tf.square(vsini)),
        c_fe - o_fe,
        0.5 * logg + 0.5 * m_h,
    ], axis=-1)
    return tf.concat([labels_23, eng], axis=-1)

def get_clean_imputed_data(h5_path, selected_labels, limit=None):
    with h5py.File(h5_path, 'r') as f:
        get_col = lambda k: f['metadata'][k]
        keys = f['metadata'].dtype.names
        raw_values = np.stack([get_col(p) for p in selected_labels], axis=1)
        bad_mask = np.zeros_like(raw_values, dtype=bool)
        for i, label in enumerate(selected_labels):
            flag_name = f"{label}_FLAG"
            if flag_name in keys:
                flg = get_col(flag_name)
                if flg.dtype.names: flg = flg[flg.dtype.names[0]]
                if flg.dtype.kind == 'V': flg = flg.view('<i4')
                bad_mask[:, i] = (flg.astype(int) != 0)
            elif label in ['TEFF', 'LOGG', 'VMICRO', 'VMACRO', 'VSINI']:
                bad_mask[:, i] = (raw_values[:, i] < -5000)
    if limit: raw_values = raw_values[:limit]; bad_mask = bad_mask[:limit]
    vals = raw_values.copy(); vals[bad_mask] = np.nan
    imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=10, initial_strategy='median')
    clean = imputer.fit_transform(vals)
    fe_h = clean[:, config.FE_H_INDEX:config.FE_H_INDEX+1]
    for idx in ABUNDANCE_INDICES: clean[:, idx] += fe_h[:, 0]
    t = clean[:, 0]
    clean = np.hstack([clean, (5040.0/(t+1e-6)).reshape(-1,1)])
    vm = clean[:, 4]; vs = clean[:, 5]
    clean = np.hstack([clean, np.sqrt(vm**2+vs**2).reshape(-1,1)])
    c = clean[:, 6]; o = clean[:, 8]
    clean = np.hstack([clean, (c-o).reshape(-1,1)])
    g = clean[:, 1]; m = clean[:, 2]
    clean = np.hstack([clean, (0.5*g+0.5*m).reshape(-1,1)])
    return clean

In [43]:
def get_err(h5_path, err_labels, limit=None):
    with h5py.File(h5_path, 'r') as f:
        get_col = lambda k: f['metadata'][k]
        keys = f['metadata'].dtype.names
        
        # Correctly handle missing error columns like VMACRO_ERR
        missing_errs = ['VMACRO_ERR', 'VMICRO_ERR', 'VSINI_ERR']
        raw = []
        for p in err_labels:
            if p in keys and p not in missing_errs:
                raw.append(get_col(p))
            else:
                # Force zero for known missing or unavailable error columns
                raw.append(np.zeros_like(get_col('TEFF_ERR')))
        raw = np.stack(raw, axis=1)
        
        bad = np.zeros_like(raw, dtype=bool)
        for i, p in enumerate(err_labels):
            # Flags are typically associated with the base label, not the error label
            base_label = p.replace('_ERR', '')
            fn = f"{base_label}_FLAG"
            if fn in keys:
                flg = get_col(fn)
                if flg.dtype.names: flg = flg[flg.dtype.names[0]]
                if flg.dtype.kind == 'V': flg = flg.view('<i4')
                bad[:, i] = (flg.astype(int) != 0)
            elif base_label in ['TEFF','LOGG','VMICRO','VMACRO','VSINI']:
                # Fallback for physical parameters if flag is missing
                raw_phys = get_col(base_label)
                bad[:, i] = (raw_phys < -5000)
                
    if limit: raw = raw[:limit]; bad = bad[:limit]
    v = raw.copy(); v[bad] = np.nan
    p95 = np.nanpercentile(v, 95, axis=0)
    return np.where(np.isnan(v), p95, v)

In [44]:
# ================================================================
# 5. MAP OPTIMIZATION (XLA)
# ================================================================

@tf.function(jit_compile=True)
def map_optimize_core(theta_unc, obs_flux, obs_ivar, fixed_abund, cnn_mu, cnn_std):
    lr_init = tf.constant(config.LR_CORE_INIT, tf.float32)
    lr_end  = tf.constant(config.LR_CORE_END, tf.float32)
    n_total = tf.constant(float(config.N_STEPS_CORE), tf.float32)
    m = tf.zeros_like(theta_unc); v = tf.zeros_like(theta_unc); theta = theta_unc
    core_cnn_mu = tf.gather(cnn_mu, CORE_INDICES, axis=1)
    core_cnn_std = tf.gather(cnn_std, CORE_INDICES, axis=1)
    
    for step in tf.range(config.N_STEPS_CORE):
        frac = tf.cast(step, tf.float32) / n_total
        lr = lr_end + 0.5 * (lr_init - lr_end) * (1.0 + tf.cos(frac * 3.14159265))
        with tf.GradientTape() as tape:
            tape.watch(theta)
            theta_phys = _core_bij.forward(theta)
            parts = []
            for i in range(23):
                if i in CORE_INDICES: ci = CORE_INDICES.index(i); parts.append(theta_phys[:, ci:ci+1])
                else: ai = ABUND_INDICES.index(i); parts.append(fixed_abund[:, ai:ai+1])
            labels_norm = (get_27_features(tf.concat(parts, axis=1)) - MEAN_TENSOR) / STD_TENSOR
            f_model=tf.cast(model_pure_fp32(labels_norm, training=False), tf.float32)
            if(len(f_model.shape)>2):
                f_model = tf.squeeze(f_model, -1)
            mask = tf.cast((obs_flux > 1e-3) & (obs_flux < 1.3) & (obs_ivar > 0.0), tf.float32)
            loss = tf.reduce_sum(0.5 * tf.reduce_sum(tf.square(obs_flux - f_model) * obs_ivar * mask, axis=1) + 
                                0.5 * tf.reduce_sum(tf.square((theta_phys - core_cnn_mu) / core_cnn_std), axis=1))
        grad = tape.gradient(loss, theta)
        t_f = tf.cast(step + 1, tf.float32)
        m = 0.9 * m + 0.1 * grad; v = 0.999 * v + 0.001 * tf.square(grad)
        m_hat = m / (1.0 - tf.pow(0.9, t_f)); v_hat = v / (1.0 - tf.pow(0.999, t_f))
        theta = theta - lr * m_hat / (tf.sqrt(v_hat) + 1e-7)
    return _core_bij.forward(theta)

@tf.function(jit_compile=True)
def map_optimize_element(theta_unc_1d, obs_flux, obs_ivar, fixed_full_23, elem_col, pixel_mask, elem_lo, elem_hi, elem_cnn_mu, elem_cnn_std, p_weight):
    lo = tf.reshape(elem_lo, [1]); hi = tf.reshape(elem_hi, [1])
    bij = tfb.Chain([tfb.Shift(lo), tfb.Scale(hi - lo), tfb.Sigmoid()])
    lr_init = tf.constant(config.LR_ELEM_INIT, tf.float32); lr_end = tf.constant(config.LR_ELEM_END, tf.float32)
    n_total = tf.constant(float(config.N_STEPS_ELEM), tf.float32)
    m = tf.zeros_like(theta_unc_1d); v = tf.zeros_like(theta_unc_1d); theta = theta_unc_1d

    for step in tf.range(config.N_STEPS_ELEM):
        frac = tf.cast(step, tf.float32) / n_total
        lr = lr_end + 0.5 * (lr_init - lr_end) * (1.0 + tf.cos(frac * 3.14159265))
        with tf.GradientTape() as tape:
            tape.watch(theta); theta_phys = bij.forward(theta)
            bc = tf.shape(fixed_full_23)[0]
            indices = tf.stack([tf.range(bc), tf.fill([bc], elem_col)], axis=1)
            full_spliced = tf.tensor_scatter_nd_update(fixed_full_23, indices, theta_phys[:, 0])
            labels_norm = (get_27_features(full_spliced) - MEAN_TENSOR) / STD_TENSOR
            f_model=tf.cast(model_pure_fp32(labels_norm, training=False), tf.float32)
            if(len(f_model.shape)>2):
                f_model = tf.squeeze(f_model, -1)
            mask = tf.cast((obs_flux > 1e-3) & (obs_flux < 1.3) & (obs_ivar > 0.0), tf.float32) * tf.reshape(pixel_mask, [1, -1])
            loss = tf.reduce_sum(0.5 * tf.reduce_sum(tf.square(obs_flux - f_model) * obs_ivar * mask, axis=1) + 
                                p_weight * 0.5 * tf.reduce_sum(tf.square((theta_phys - elem_cnn_mu) / elem_cnn_std), axis=1))
        grad = tape.gradient(loss, theta); t_f = tf.cast(step+1, tf.float32)
        m = 0.9 * m + 0.1 * grad; v = 0.999 * v + 0.001 * tf.square(grad)
        m_hat = m / (1.0 - tf.pow(0.9, t_f)); v_hat = v / (1.0 - tf.pow(0.999, t_f))
        theta = theta - lr * m_hat / (tf.sqrt(v_hat) + 1e-7)
    return bij.forward(theta)[:, 0]

@tf.function(jit_compile=True)
def map_optimize_joint(theta_unc, obs_flux, obs_ivar, cnn_mu, cnn_std):
    """Batched Adam MAP for all 23 parameters simultaneously."""
    lr_init = tf.constant(config.LR_CORE_INIT, tf.float32)
    lr_end  = tf.constant(config.LR_CORE_END, tf.float32)
    n_total = tf.constant(float(config.N_STEPS_CORE), tf.float32)
    m = tf.zeros_like(theta_unc); v = tf.zeros_like(theta_unc); theta = theta_unc

    for step in tf.range(config.N_STEPS_CORE):
        frac = tf.cast(step, tf.float32) / n_total
        lr = lr_end + 0.5 * (lr_init - lr_end) * (1.0 + tf.cos(frac * 3.14159265))
        with tf.GradientTape() as tape:
            tape.watch(theta)
            theta_phys = _joint_bij.forward(theta)
            labels_norm = (get_27_features(theta_phys) - MEAN_TENSOR) / STD_TENSOR
            f_model=tf.cast(model_pure_fp32(labels_norm, training=False), tf.float32)
            if(len(f_model.shape)>2):
                f_model = tf.squeeze(f_model, -1)
            mask = tf.cast((obs_flux > 1e-3) & (obs_flux < 1.3) & (obs_ivar > 0.0), tf.float32)
            chi2 = tf.reduce_sum(tf.square(obs_flux - f_model) * obs_ivar * mask, axis=1)
            prior = tf.reduce_sum(tf.square((theta_phys - cnn_mu) / cnn_std), axis=1)
            loss = tf.reduce_sum(0.5 * chi2 + config.PRIOR_WEIGHT * 0.5 * prior)
        grad = tape.gradient(loss, theta); t_f = tf.cast(step + 1, tf.float32)
        m = 0.9 * m + 0.1 * grad; v = 0.999 * v + 0.001 * tf.square(grad)
        m_hat = m / (1.0 - tf.pow(0.9, t_f)); v_hat = v / (1.0 - tf.pow(0.999, t_f))
        theta = theta - lr * m_hat / (tf.sqrt(v_hat) + 1e-7)
    return _joint_bij.forward(theta)

In [48]:
# ================================================================
# 6. LAPLACE UNCERTAINTY ESTIMATION
# ================================================================

@tf.function()
def compute_uncertainties_core(theta_unc_batch, obs_flux, obs_ivar, fixed_abund, cnn_mu, cnn_std):
    n_params = 9
    core_cnn_mu = tf.gather(cnn_mu, CORE_INDICES, axis=1)
    core_cnn_std = tf.gather(cnn_std, CORE_INDICES, axis=1)
    # BROADCAST FIX: Ensure bounds are explicitly 9D and gathered
    c_low  = tf.gather(bounds_low, CORE_INDICES)
    c_high = tf.gather(bounds_high, CORE_INDICES)
    
    def single_star_hessian(args):
        theta_unc, flux, ivar, f_abund, c_mu, c_std = args
        with tf.GradientTape() as outer_tape:
            outer_tape.watch(theta_unc)
            with tf.GradientTape() as inner_tape:
                inner_tape.watch(theta_unc)
                theta_phys = _core_bij.forward(theta_unc)
                parts = []
                for i in range(23):
                    if i in CORE_INDICES: ci = CORE_INDICES.index(i); parts.append(tf.reshape(theta_phys[ci], [1]))
                    else: ai = ABUND_INDICES.index(i); parts.append(tf.reshape(f_abund[ai], [1]))
                labels_norm = (get_27_features(tf.expand_dims(tf.concat(parts, axis=0), 0)) - MEAN_TENSOR) / STD_TENSOR
                f_model=tf.cast(model_pure_fp32(labels_norm, training=False), tf.float32)
                if(len(f_model.shape)>2):
                    f_model = tf.squeeze(f_model, -1)
                mask = tf.cast((flux > 1e-3) & (flux < 1.3) & (ivar > 0.0), tf.float32)
                loss = 0.5 * tf.reduce_sum(tf.square(flux - f_model) * ivar * mask) + 0.5 * tf.reduce_sum(tf.square((theta_phys - c_mu) / c_std))
            grad = inner_tape.gradient(loss, theta_unc)
        hess = outer_tape.jacobian(grad, theta_unc)
        cov = tf.linalg.inv(hess + tf.eye(n_params) * 1e-6)
        var = tf.abs(tf.linalg.diag_part(cov))
        s = tf.nn.sigmoid(theta_unc); jac = s * (1.0 - s) * (c_high - c_low)
        return tf.sqrt(var) * jac
    return tf.vectorized_map(single_star_hessian, (theta_unc_batch, obs_flux, obs_ivar, fixed_abund, core_cnn_mu, core_cnn_std))

@tf.function()
def compute_uncertainties_element(theta_unc_1d, obs_flux, obs_ivar, fixed_full_23, elem_col, pixel_mask, elem_lo, elem_hi, elem_cnn_mu, elem_cnn_std, p_weight):
    lo = tf.cast(elem_lo, tf.float32); hi = tf.cast(elem_hi, tf.float32)
    bij = tfb.Chain([tfb.Shift(lo), tfb.Scale(hi - lo), tfb.Sigmoid()])
    def single_star_hessian(args):
        theta_unc, flux, ivar, f_23, c_mu, c_std = args
        with tf.GradientTape() as tape:
            tape.watch(theta_unc)
            with tf.GradientTape() as inner_tape:
                inner_tape.watch(theta_unc); theta_phys = bij.forward(theta_unc)
                indices = tf.constant([[0, elem_col]])
                full_spliced = tf.tensor_scatter_nd_update(tf.expand_dims(f_23, 0), indices, tf.reshape(theta_phys, [1]))
                labels_norm = (get_27_features(full_spliced) - MEAN_TENSOR) / STD_TENSOR
                f_model=tf.cast(model_pure_fp32(labels_norm, training=False), tf.float32)
                if(len(f_model.shape)>2):
                    f_model = tf.squeeze(f_model, -1)
                mask = tf.cast((flux > 1e-3) & (flux < 1.3) & (ivar > 0.0), tf.float32) * pixel_mask
                loss = 0.5 * tf.reduce_sum(tf.square(flux - f_model) * ivar * mask) + p_weight * 0.5 * tf.reduce_sum(tf.square((theta_phys - c_mu) / c_std))
            grad = inner_tape.gradient(loss, theta_unc)
        hess = tape.gradient(grad, theta_unc); var = 1.0 / (tf.abs(hess) + 1e-6)
        s = tf.nn.sigmoid(theta_unc); jac = s * (1.0 - s) * (hi - lo)
        return tf.sqrt(var) * jac
    return tf.vectorized_map(single_star_hessian, (theta_unc_1d, obs_flux, obs_ivar, fixed_full_23, elem_cnn_mu, elem_cnn_std))

@tf.function()
def compute_uncertainties_joint(theta_unc_batch, obs_flux, obs_ivar, cnn_mu, cnn_std):
    """Laplace approx for Joint MAP (23D)."""
    n_params = 23
    def single_star_hessian(args):
        theta_unc, flux, ivar, c_mu, c_std = args
        with tf.GradientTape() as outer_tape:
            outer_tape.watch(theta_unc)
            with tf.GradientTape() as inner_tape:
                inner_tape.watch(theta_unc)
                theta_phys = _joint_bij.forward(theta_unc)
                labels_norm = (get_27_features(tf.expand_dims(theta_phys, 0)) - MEAN_TENSOR) / STD_TENSOR
                f_model=tf.cast(model_pure_fp32(labels_norm, training=False), tf.float32)
                if(len(f_model.shape)>2):
                    f_model = tf.squeeze(f_model, -1)
                mask = tf.cast((flux > 1e-3) & (flux < 1.3) & (ivar > 0.0), tf.float32)
                chi2 = tf.reduce_sum(tf.square(flux - f_model) * ivar * mask)
                prior = tf.reduce_sum(tf.square((theta_phys - c_mu) / c_std))
                loss = 0.5 * chi2 + config.PRIOR_WEIGHT * 0.5 * prior
            grad = inner_tape.gradient(loss, theta_unc)
        hess = outer_tape.jacobian(grad, theta_unc)
        cov = tf.linalg.inv(hess + tf.eye(n_params) * 1e-6)
        var = tf.abs(tf.linalg.diag_part(cov))
        s = tf.nn.sigmoid(theta_unc); jac = s * (1.0 - s) * (bounds_high - bounds_low)
        return tf.sqrt(var) * jac
    return tf.vectorized_map(single_star_hessian, (theta_unc_batch, obs_flux, obs_ivar, cnn_mu, cnn_std))

In [51]:
# ================================================================
# 7. MAIN INFERENCE PIPELINE
# ================================================================

def run_inference_pipeline(test_indices, true_labels_norm, aspcap_errors, flux_array, ivar_array):
    n_stars = len(test_indices)
    true_labels_physical = (true_labels_norm[:, :23] * STD_TENSOR[:23] + MEAN_TENSOR[:23])
    
    results = {
        'global_indices': [], 'true_labels': [], 'inferred_labels': [],
        'inferred_errors': [], 'aspcap_errors': [], 'wall_seconds': []
    }
    
    print(f"\n  Starting Inference on {n_stars} stars (Batch Size: {config.BATCH_SIZE_STARS})")
    
    for batch_start in range(0, n_stars, config.BATCH_SIZE_STARS):
        batch_local = list(range(batch_start, min(batch_start + config.BATCH_SIZE_STARS, n_stars)))
        actual = len(batch_local)
        
        b_flux = flux_array[batch_local].astype(np.float32)
        b_ivar = ivar_array[batch_local].astype(np.float32)
        cnn_mu, cnn_sig = cnn_predict_physical(b_flux)

        if actual < config.BATCH_SIZE_STARS:
            pad = config.BATCH_SIZE_STARS - actual
            b_flux = np.concatenate([b_flux, np.zeros((pad, 8575), np.float32)])
            b_ivar = np.concatenate([b_ivar, np.zeros((pad, 8575), np.float32)])
            cnn_mu = np.concatenate([cnn_mu, np.tile(cnn_mu[:1], (pad, 1))])
            cnn_sig = np.concatenate([cnn_sig, np.tile(cnn_sig[:1], (pad, 1))])

        obs_flux_tf, obs_ivar_tf = tf.constant(b_flux), tf.constant(b_ivar)
        cnn_mu_tf, cnn_sig_tf = tf.constant(cnn_mu), tf.constant(cnn_sig)

        # -- Stage 1: Core --
        t0 = time.time()
        # BROADCAST FIX: Explicitly gather bounds for clipping
        local_low  = tf.gather(bounds_low, CORE_INDICES)[None, :]
        local_high = tf.gather(bounds_high, CORE_INDICES)[None, :]
        core_init_phys = tf.clip_by_value(tf.gather(cnn_mu_tf, CORE_INDICES, axis=1), local_low + 1e-3, local_high - 1e-3)
        
        fixed_abund_tf = tf.gather(cnn_mu_tf, ABUND_INDICES, axis=1)
        core_result = map_optimize_core(_core_inv.forward(core_init_phys), obs_flux_tf, obs_ivar_tf, fixed_abund_tf, cnn_mu_tf, cnn_sig_tf)
        core_errs = compute_uncertainties_core(_core_inv.forward(core_result), obs_flux_tf, obs_ivar_tf, fixed_abund_tf, cnn_mu_tf, cnn_sig_tf)
        t_core=time.time()-t0
        # -- Stage 2: Elements --
        t0 = time.time()
        fixed_23 = cnn_mu.copy()
        core_np, core_err_np = core_result.numpy(), core_errs.numpy()
        for ci, gi in enumerate(CORE_INDICES): fixed_23[:, gi] = core_np[:, ci]
        
        elem_results = np.zeros((config.BATCH_SIZE_STARS, 23), np.float32)
        elem_errors  = np.zeros((config.BATCH_SIZE_STARS, 23), np.float32)
        for ci, gi in enumerate(CORE_INDICES):
            elem_results[:, gi] = core_np[:, ci]; elem_errors[:, gi] = core_err_np[:, ci]

        for ai, elem_idx in enumerate(ABUND_INDICES):
            lo, hi = lower_bounds[elem_idx], upper_bounds[elem_idx]
            inv_bij = tfb.Invert(tfb.Chain([tfb.Shift(tf.reshape(lo, [1])), tfb.Scale(tf.reshape(hi-lo, [1])), tfb.Sigmoid()]))
            e_init = tf.clip_by_value(cnn_mu_tf[:, elem_idx:elem_idx+1], lo+1e-3, hi-1e-3)
            e_res = map_optimize_element(inv_bij.forward(e_init), obs_flux_tf, obs_ivar_tf, tf.constant(fixed_23, tf.float32), elem_idx, ELEMENT_PIXEL_MASKS[elem_idx], lo, hi, cnn_mu_tf[:, elem_idx:elem_idx+1], cnn_sig_tf[:, elem_idx:elem_idx+1], ELEMENT_PRIOR_WEIGHTS[elem_idx])
            e_err = compute_uncertainties_element(inv_bij.forward(e_res), obs_flux_tf, obs_ivar_tf, tf.constant(fixed_23, tf.float32), elem_idx, ELEMENT_PIXEL_MASKS[elem_idx], lo, hi, cnn_mu_tf[:, elem_idx:elem_idx+1], cnn_sig_tf[:, elem_idx:elem_idx+1], ELEMENT_PRIOR_WEIGHTS[elem_idx])
            elem_results[:, elem_idx] = e_res.numpy(); elem_errors[:, elem_idx] = e_err.numpy(); fixed_23[:, elem_idx] = e_res.numpy()
        t_elem = time.time() - t0

        for b in range(actual):
            results['global_indices'].append(int(test_indices[batch_local[b]]))
            results['true_labels'].append(true_labels_physical[batch_local[b]])
            results['inferred_labels'].append(elem_results[b])
            results['inferred_errors'].append(elem_errors[b])
            results['aspcap_errors'].append(aspcap_errors[batch_local[b]])
            results['wall_seconds'].append((t_core + t_elem) / actual)
        print(f"  [{min(batch_start + config.BATCH_SIZE_STARS, n_stars):>4}/{n_stars}] Batch complete.")

    return results

def select_stratified_test_sample(h5_path, test_start, test_end, target_n=500, logg_bins=5, teff_bins=10, mh_bins=10):
    """Selects stars across a 3D grid of logg, teff, and m_h for balanced testing."""
    with h5py.File(h5_path, 'r') as f:
        meta = f['metadata']
        teff = meta['TEFF'][test_start:test_end].astype(np.float32)
        logg = meta['LOGG'][test_start:test_end].astype(np.float32)
        m_h  = meta['M_H'][test_start:test_end].astype(np.float32)
        snr  = meta['SNR'][test_start:test_end].astype(np.float32)

    SENTINEL = -5000.0
    valid = (teff > SENTINEL) & (logg > SENTINEL) & (m_h > SENTINEL)

    def percentile_edges(values, mask, n_bins):
        pts = np.linspace(0, 100, n_bins + 1)
        edges = np.percentile(values[mask], pts)
        edges[0] -= 1e-6; edges[-1] += 1e-6
        return edges

    logg_edges = percentile_edges(logg, valid, logg_bins)
    teff_edges = percentile_edges(teff, valid, teff_bins)
    mh_edges   = percentile_edges(m_h, valid, mh_bins)

    logg_bin = np.clip(np.digitize(logg, logg_edges) - 1, 0, logg_bins - 1)
    teff_bin = np.clip(np.digitize(teff, teff_edges) - 1, 0, teff_bins - 1)
    mh_bin   = np.clip(np.digitize(m_h, mh_edges) - 1, 0, mh_bins - 1)

    local_indices_selected = []
    bin_summary = []
    for i in range(logg_bins):
        for j in range(teff_bins):
            for k in range(mh_bins):
                in_bin = valid & (logg_bin == i) & (teff_bin == j) & (mh_bin == k)
                idx = np.where(in_bin)[0]
                if len(idx) == 0: continue
                # Pick the star with median SNR in this bin
                median_snr = np.median(snr[idx])
                closest = idx[np.argmin(np.abs(snr[idx] - median_snr))]
                local_indices_selected.append(closest)
                bin_summary.append({'n_stars': len(idx)})

    local_indices_selected = np.array(local_indices_selected)
    n_from_bins = len(local_indices_selected)

    if n_from_bins < target_n:
        shortfall = target_n - n_from_bins
        remaining = np.array([i for i in np.where(valid)[0] if i not in set(local_indices_selected.tolist())])
        rng = np.random.default_rng(config.RANDOM_SEED)
        extra = rng.choice(remaining, size=min(shortfall, len(remaining)), replace=False)
        local_indices_selected = np.concatenate([local_indices_selected, extra])
    elif n_from_bins > target_n:
        # If we have too many, keep those from the most populous bins first
        occupancies = np.array([b['n_stars'] for b in bin_summary])
        order = np.argsort(occupancies)[::-1] # descending
        local_indices_selected = local_indices_selected[order[:target_n]]

    return np.sort(local_indices_selected)

In [ ]:
# ================================================================
# 8. MAIN EXECUTION
# ================================================================

if __name__ == "__main__":
  
    # Load Stats
    with np.load(config.STATS_PATH) as data:
        MEAN_TENSOR = data['mean'].astype(np.float32)
        STD_TENSOR  = data['std'].astype(np.float32)

    # Rebuild Model in FP32
    model_raw = tf.keras.models.load_model(config.MODEL_PATH)
    cfg_str = json.dumps(model_raw.get_config()).replace('"float16"', '"float32"').replace('"mixed_float16"', '"float32"')
    model_pure_fp32 = model_raw.__class__.from_config(json.loads(cfg_str))
    model_pure_fp32.set_weights(model_raw.get_weights())

    # Warmstart Predictor
    cnn_predictor = tf.keras.models.load_model(config.CNN_MODEL_PATH)
    with np.load(config.CNN_STATS_PATH) as cs:
        CNN_LABEL_MEAN, CNN_LABEL_STD = cs['mean'].astype(np.float32), cs['std'].astype(np.float32)

    def cnn_predict_physical(flux):
        mu_n, rlv = cnn_predictor(tf.constant(flux, tf.float32), training=False)
        mu_p = mu_n.numpy() * CNN_LABEL_STD + CNN_LABEL_MEAN
        std_p = np.sqrt(tf.nn.softplus(rlv).numpy() + 1e-4) * CNN_LABEL_STD
        return mu_p.astype(np.float32), np.maximum(std_p, 0.01 * CNN_LABEL_STD).astype(np.float32)

    # Element Masks and Weights
    raw_mask = np.load(config.APOGEE_MASK_PATH)
    ELEMENT_PIXEL_MASKS = {}
    ELEMENT_PRIOR_WEIGHTS = {}
    for idx in ABUND_INDICES:
        m = (raw_mask[:, idx] > 0.01).astype(np.float32)
        ELEMENT_PIXEL_MASKS[idx] = tf.constant(m if m.sum() > 10 else np.ones(8575, np.float32), tf.float32)
        pw = np.clip(500.0 / max(m.sum(), 10.0), 0.1, 20.0)
        ELEMENT_PRIOR_WEIGHTS[idx] = tf.constant(pw, tf.float32)

    # Data Loading
    print(f"Reading H5 from {config.H5_PATH}...")
    with h5py.File(config.H5_PATH, 'r') as f:
        flux_all = f['flux'][config.TEST_START:config.TEST_END]
        ivar_all = f['ivar'][config.TEST_START:config.TEST_END]
    
    labels_all = get_clean_imputed_data(config.H5_PATH, config.SELECTED_LABELS, limit=config.TEST_END)
    labels_all = ((labels_all - MEAN_TENSOR) / STD_TENSOR)[config.TEST_START:config.TEST_END]
    errs_all   = get_err(config.H5_PATH, config.ERRORS, limit=config.TEST_END)[config.TEST_START:config.TEST_END]

    # Stratified Sample
    print(f"Selecting {config.TOTAL_STARS} test stars (stratified)...")
    local_indices = select_stratified_test_sample(
        config.H5_PATH, config.TEST_START, config.TEST_END, 
        target_n=config.TOTAL_STARS
    )

In [52]:
    
    # Run Inference
    final_results = run_inference_pipeline(
        local_indices, 
        labels_all[local_indices], 
        errs_all[local_indices], 
        flux_all[local_indices], 
        ivar_all[local_indices]
    )
    
    # Enriched Reporting
    trues  = np.array(final_results['true_labels'])
    preds  = np.array(final_results['inferred_labels'])
    errors = np.array(final_results['inferred_errors'])
    asp_err = np.array(final_results['aspcap_errors'])
    residuals = preds - trues
    
    print(f"\n{'='*70}")
    print(f"  PRISM FINAL ENRICHED DIAGNOSTICS (N={len(local_indices)})")
    print(f"{'='*70}")
    hdr = f"{'Label':<12} {'Bias':>8} {'MAD':>8} {'RMSE':>8} {'Mod-σ':>8} {'Ratio':>6}"
    print(hdr)
    print("-" * len(hdr))
    
    for j, name in enumerate(config.SELECTED_LABELS):
        r, ie, ae = residuals[:, j], errors[:, j], asp_err[:, j]
        bias = np.median(r)
        mad  = np.median(np.abs(r - bias))
        rmse = np.sqrt(np.mean(r**2))
        avg_ie = np.sqrt(np.mean(ie**2))
        # Ratio of ASPCAP reported error to PRISM inferred error
        ratio = np.median(ae / np.where(ie > 1e-10, ie, 1e-10))
        print(f"{name:<12} {bias:>8.3f} {mad:>8.3f} {rmse:>8.3f} {avg_ie:>8.3f} {ratio:>6.1f}x")
    
    print("=" * len(hdr))
    
    # Save Final Output
    save_path = "/kaggle/working/prism_final_results.npz"
    np.savez(save_path, **{k: np.array(v) for k, v in final_results.items()}, label_names=config.SELECTED_LABELS)
    print(f"\n  Success! Results saved to {save_path}")


  Starting Inference on 500 stars (Batch Size: 32)
  [  32/500] Batch complete.
  [  64/500] Batch complete.
  [  96/500] Batch complete.
  [ 128/500] Batch complete.
  [ 160/500] Batch complete.
  [ 192/500] Batch complete.
  [ 224/500] Batch complete.
  [ 256/500] Batch complete.
  [ 288/500] Batch complete.
  [ 320/500] Batch complete.
  [ 352/500] Batch complete.
  [ 384/500] Batch complete.
  [ 416/500] Batch complete.
  [ 448/500] Batch complete.
  [ 480/500] Batch complete.
  [ 500/500] Batch complete.

  PRISM FINAL ENRICHED DIAGNOSTICS (N=500)
Label            Bias      MAD     RMSE    Mod-σ  Ratio
-------------------------------------------------------
TEFF           80.915   61.873  181.889    2.817    3.7x
LOGG            0.208    0.125    0.366    0.006    4.7x
M_H             0.083    0.031    0.158    0.002    3.1x
VMICRO          0.017    0.125    0.334    0.005    0.0x
VMACRO         -0.285    0.110    0.930    0.005    0.0x
VSINI           1.760    2.612    5.479    